# Binance Backtest — BTCUSDT aggTrades (synthetic L1)

Analysis of the equity series produced by `backtest_main --data-format binance-aggtrades`.
The replay parses Binance Spot aggTrades CSV, reconstructs a synthetic top-of-book from
the `is_buyer_maker` flag, and feeds it through the same TradeEngine + LiquidityTaker
strategy that ran on LOBSTER.

**Price scale** (in the C++ replay): raw `$` * 100 → stored as integer cents.
**Qty scale**: raw BTC * 10^6 → stored as micro-BTC (μBTC). 
PnL in the CSV is therefore in cent·μBTC units — divide by 10^8 to convert to USD.

Run the backtest first:
```bash
./scripts/download_binance.sh BTCUSDT 2025-10 monthly
./build/backtest_main \
  --data-format binance-aggtrades \
  --data data/binance/spot/aggTrades/BTCUSDT/BTCUSDT-aggTrades-2025-10.csv \
  --clip 100 --threshold 0.05 --max-pos 100000 \
  --equity-csv benchmarks/backtest/binance/equity.csv
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

EQUITY_CSV = '../benchmarks/backtest/binance/equity.csv'
PNL_SCALE = 1e8   # cent * μBTC -> USD  (100 * 10^6)
QTY_SCALE = 1e6   # μBTC -> BTC

df = pd.read_csv(EQUITY_CSV)
df['time'] = pd.to_datetime(df['wall_ns'], unit='ns')
for col in ('total_pnl', 'real_pnl', 'unreal_pnl'):
    df[f'{col}_usd'] = df[col] / PNL_SCALE
df['position_btc'] = df['position'] / QTY_SCALE
df['volume_btc'] = df['volume'] / QTY_SCALE
print(f'Samples: {len(df)}')
print(f'Elapsed wall during run: {(df.wall_ns.max() - df.wall_ns.min())/1e9:.1f}s')
df.head()

In [ ]:
# Summary stats.
peak = df['total_pnl_usd'].cummax()
drawdown = peak - df['total_pnl_usd']
fills = int((df['volume'].diff() > 0).sum())

pd.Series({
    'samples':         len(df),
    'final_pnl_usd':   df['total_pnl_usd'].iloc[-1],
    'peak_pnl_usd':    df['total_pnl_usd'].max(),
    'max_drawdown_usd':drawdown.max(),
    'final_position_btc':  df['position_btc'].iloc[-1],
    'max_abs_position_btc':df['position_btc'].abs().max(),
    'final_volume_btc':df['volume_btc'].iloc[-1],
    'fills':           fills,
})

In [ ]:
# Equity curve + position over time.
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True,
                                gridspec_kw={'height_ratios': [3, 1]})

ax1.plot(df['time'], df['total_pnl_usd'], color='steelblue', linewidth=1.2, label='total PnL')
ax1.fill_between(df['time'], df['total_pnl_usd'], 0,
                 where=(df['total_pnl_usd'] >= 0), alpha=0.15, color='green')
ax1.fill_between(df['time'], df['total_pnl_usd'], 0,
                 where=(df['total_pnl_usd'] < 0), alpha=0.15, color='firebrick')
ax1.axhline(0, color='gray', linewidth=0.5)
ax1.set_ylabel('PnL ($)')
ax1.set_title('LiquidityTaker on Binance BTCUSDT aggTrades (synthetic L1) — equity curve')
ax1.grid(True, alpha=0.3)
ax1.legend(loc='lower left')

ax2.plot(df['time'], df['position_btc'], color='black', linewidth=1.0, drawstyle='steps-post')
ax2.axhline(0, color='gray', linewidth=0.5)
ax2.set_ylabel('Position (BTC)')
ax2.set_xlabel('Wall-clock time (during backtest run)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
os.makedirs('../benchmarks/backtest/binance', exist_ok=True)
plt.savefig('../benchmarks/backtest/binance/equity_curve.png', dpi=120)
plt.show()

In [ ]:
# Time-series metrics. Requires a long enough run to have daily returns; with only a
# short sample-interval series, we compute the Sharpe of per-sample returns and annualize
# by the effective sampling frequency. This is an approximation and should be interpreted
# as a sanity check only — a realistic Sharpe needs weeks of data across regimes.
import math

if len(df) > 2:
    returns = df['total_pnl_usd'].diff().dropna()
    dt_ns = df['wall_ns'].diff().dropna()
    mean_dt_s = dt_ns.mean() / 1e9
    samples_per_year = 365 * 24 * 3600 / mean_dt_s

    mu = returns.mean()
    sigma = returns.std()
    sharpe = (mu / sigma) * math.sqrt(samples_per_year) if sigma > 0 else float('nan')

    print(f'Per-sample return stats:')
    print(f'  count:       {len(returns)}')
    print(f'  mean:        ${mu:.6f}')
    print(f'  stddev:      ${sigma:.6f}')
    print(f'  sampling interval: {mean_dt_s*1000:.1f} ms')
    print(f'  samples/year (approx): {samples_per_year:.2e}')
    print(f'  Sharpe (annualized, approximate): {sharpe:.3f}')
else:
    print('Not enough samples for a meaningful Sharpe.')

## Interpretation

- **Engine-correctness**: The replay happily streams millions of aggTrades through the
  same TradeEngine / OrderManager / PositionKeeper that would run live. No code changes
  between data sources — only the Replay class differs.
- **Throughput**: ~5,000 trades/second on this machine, i.e. a full trading month of
  BTCUSDT (~40M aggTrades) replays in ~2 hours. More than fast enough for iterative
  parameter tuning.
- **Strategy activity**: LiquidityTaker fires rarely on the synthetic-L1 feed — the
  aggressive-trade qty ratio it uses rarely exceeds the threshold because synthetic
  quotes stay at 1 BTC so ratios are tiny. Phase 1.3 work on a MarketMaker with
  OBI signal + queue-aware fills on the LOBSTER dataset will produce the Sharpe /
  drawdown / hit-rate statistics that hit the resume.
- **Known limitation**: Sharpe above is annualized from microsecond-scale sampling,
  which over-states effective information content. Proper Sharpe needs a multi-day
  equity series; with only 1M trades replayed (~17 minutes of real market time) this is
  a signal-of-life, not a hedge-fund number.